# 01 — Artifact Removal

This notebook mirrors `python/01_artifact_removal.py` with richer documentation and inline visualisations.

## What this step does

Physiologically implausible values are **set to NaN** (not dropped) in three signals:

| Signal | Rule |
|--------|------|
| `hrate_rest_fitb` | Outside [40, 200] bpm, or flagged as non-wear by `flag_nonwear()` |
| `steps_total` | Negative values, or within-participant-day spikes > mean + 5 SD |
| `min_slp` | Outside [0, 120] minutes per 2-hr slot |

**Input:** `dev/data/fitbit-summaries/activity_120m.parquet`  
**Output:** `data/processed/clean/clean_data.parquet`

**Data structure:** each row is one participant × session × day × 2-hour slot (12 slots per day, epochs starting at 00:00, 02:00, …, 22:00).

In [ ]:
import sys
from pathlib import Path

# Allow imports from python/ when running notebook from python/notebooks/
ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))

import logging

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from config import (
    RAW_PARQUET, CLEAN_PARQUET, LOGS_DIR,
    SESSIONS,
    COL_ID, COL_SESSION, COL_DAY, COL_START, COL_WKND,
    COL_HR, COL_SLEEP, COL_STEPS,
    HR_MIN, HR_MAX, STEPS_SPIKE_SD, SLEEP_MIN, SLEEP_MAX,
)
from utils.wear_quality import flag_nonwear

LOGS_DIR.mkdir(parents=True, exist_ok=True)
CLEAN_PARQUET.parent.mkdir(parents=True, exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)s  %(message)s",
    handlers=[
        logging.FileHandler(LOGS_DIR / "01_artifact_removal.log"),
        logging.StreamHandler(),
    ],
)
log = logging.getLogger(__name__)
print(f"Config loaded. Active sessions: {SESSIONS}")
print(f"HR range: [{HR_MIN}, {HR_MAX}] bpm | Steps spike SD: {STEPS_SPIKE_SD} | Sleep range: [{SLEEP_MIN}, {SLEEP_MAX}] min")

## Load raw data

The source parquet contains all sessions. We filter to the two active waves (`ses-02A` yr2 and `ses-06A` yr6).

In [ ]:
log.info(f"Loading {RAW_PARQUET}")
raw = pd.read_parquet(RAW_PARQUET)
log.info(f"  Raw rows: {len(raw):,}")

df = raw[raw[COL_SESSION].isin(SESSIONS)].copy()
log.info(f"  After session filter {SESSIONS}: {len(df):,} rows, {df[COL_ID].nunique():,} participants")

print(f"\nShape after session filter: {df.shape}")
print(f"Participants: {df[COL_ID].nunique():,}")
print(f"\nSession counts:")
print(df[COL_SESSION].value_counts().to_string())
print(f"\nColumn dtypes:")
print(df.dtypes)

In [ ]:
# Parse timestamps and derive start_hour (0, 2, 4, ..., 22)
df[COL_START] = pd.to_datetime(df[COL_START])
df["start_hour"] = df[COL_START].dt.hour

print("start_hour value counts (should be 12 bins: 0,2,4,...,22):")
print(df["start_hour"].value_counts().sort_index().to_string())

## HR artifact removal

Two passes:
1. **Range filter** — any HR outside [40, 200] bpm is set to NaN.
2. **Non-wear detection** — `flag_nonwear()` (from `utils/wear_quality.py`) identifies slots where the device was likely not worn (based on extended runs of very low/zero HR or steps). Those slots are also set to NaN.

Values are **not dropped** — NaN rows are preserved so the complete time grid remains intact for downstream imputation.

In [ ]:
# Capture pre-removal distribution for plotting
hr_before = df[COL_HR].dropna().copy()
n_before_hr = df[COL_HR].notna().sum()

# Range filter
out_of_range = (df[COL_HR] < HR_MIN) | (df[COL_HR] > HR_MAX)
df.loc[out_of_range, COL_HR] = np.nan
n_range_removed = int(out_of_range.sum())

# Non-wear detection
nonwear = flag_nonwear(df)
n_nonwear_removed = int(nonwear.sum())
df.loc[nonwear, COL_HR] = np.nan
# TODO: sleep_stages — non-wear detection is a stub (0 slots flagged).
#       Implement once raw per-minute stage data is available.

n_removed_hr = n_before_hr - df[COL_HR].notna().sum()
log.info(f"  HR removed: {n_removed_hr:,} ({n_removed_hr / max(n_before_hr, 1):.1%})")

print(f"HR values before:        {n_before_hr:,}")
print(f"  Out-of-range removed:  {n_range_removed:,}")
print(f"  Non-wear flagged:      {n_nonwear_removed:,}")
print(f"Total HR removed:        {n_removed_hr:,} ({n_removed_hr / max(n_before_hr, 1):.1%})")

In [ ]:
# Visualise HR distribution before vs after artifact removal
hr_after = df[COL_HR].dropna().copy()

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=False)

axes[0].hist(hr_before, bins=80, color="steelblue", edgecolor="white", linewidth=0.3)
axes[0].axvline(HR_MIN, color="red", linestyle="--", label=f"HR_MIN={HR_MIN}")
axes[0].axvline(HR_MAX, color="red", linestyle="--", label=f"HR_MAX={HR_MAX}")
axes[0].set_title(f"HR before removal  (n={len(hr_before):,})")
axes[0].set_xlabel("Resting HR (bpm)")
axes[0].set_ylabel("Count")
axes[0].legend()

axes[1].hist(hr_after, bins=80, color="darkorange", edgecolor="white", linewidth=0.3)
axes[1].set_title(f"HR after removal  (n={len(hr_after):,})")
axes[1].set_xlabel("Resting HR (bpm)")
axes[1].set_ylabel("Count")

plt.tight_layout()
plt.show()

## Steps artifact removal

Two rules:
1. **Negative values** — physiologically impossible, set to NaN.
2. **Within-day spikes** — for each (participant, session, day), any slot with steps > mean + 5 SD is considered a recording error and set to NaN. This is computed per-day to be sensitive to individual baseline differences.

In [ ]:
steps_before = df[COL_STEPS].dropna().copy()
n_before_steps = df[COL_STEPS].notna().sum()

neg = df[COL_STEPS] < 0

def _spike_mask(grp: pd.DataFrame) -> pd.Series:
    mu = grp[COL_STEPS].mean()
    sd = grp[COL_STEPS].std()
    if pd.isna(sd) or sd == 0:
        return pd.Series(False, index=grp.index)
    return grp[COL_STEPS] > mu + STEPS_SPIKE_SD * sd

spikes = (
    df.groupby([COL_ID, COL_SESSION, COL_DAY], group_keys=False)
    .apply(_spike_mask)
)
n_neg    = int(neg.sum())
n_spikes = int(spikes.sum())
df.loc[neg | spikes, COL_STEPS] = np.nan

n_removed_steps = n_before_steps - df[COL_STEPS].notna().sum()
log.info(f"  Steps removed: {n_removed_steps:,} ({n_removed_steps / max(n_before_steps, 1):.1%})")

print(f"Steps values before:     {n_before_steps:,}")
print(f"  Negative values:       {n_neg:,}")
print(f"  Within-day spikes:     {n_spikes:,}")
print(f"Total steps removed:     {n_removed_steps:,} ({n_removed_steps / max(n_before_steps, 1):.1%})")

## Sleep artifact removal

`min_slp` represents minutes of sleep recorded within a 2-hour epoch. Physical bounds are [0, 120] minutes.  

> **TODO (future):** Once raw per-minute Fitbit sleep stage data become available, add stage-code validation (e.g., reject invalid stage transitions) here.

In [ ]:
# TODO: sleep_stages — add Fitbit sleep stage code validation here once
#       raw per-minute data is available.
n_before_sleep = df[COL_SLEEP].notna().sum()

out_of_range_slp = (df[COL_SLEEP] < SLEEP_MIN) | (df[COL_SLEEP] > SLEEP_MAX)
n_slp_removed = int(out_of_range_slp.sum())
df.loc[out_of_range_slp, COL_SLEEP] = np.nan

log.info(f"  Sleep removed: {n_slp_removed:,} ({n_slp_removed / max(n_before_sleep, 1):.1%})")

print(f"Sleep values before:     {n_before_sleep:,}")
print(f"  Out-of-range removed:  {n_slp_removed:,} ({n_slp_removed / max(n_before_sleep, 1):.2%})")

## Summary and save

Before writing the clean parquet we print a tidy summary of NaN counts per signal per session. This helps verify that artifact removal rates are reasonable (e.g., not removing >20% of any signal).

In [ ]:
# Summary table: NaN counts per signal per session
summary_rows = []
for sess in SESSIONS:
    sub = df[df[COL_SESSION] == sess]
    n = len(sub)
    for sig in [COL_HR, COL_STEPS, COL_SLEEP]:
        nan_count = sub[sig].isna().sum()
        summary_rows.append({
            "session":      sess,
            "signal":       sig,
            "total_rows":   n,
            "nan_count":    nan_count,
            "nan_pct":      f"{nan_count / n:.1%}",
            "valid_count":  n - nan_count,
        })

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))

# Save clean parquet
cols = [COL_ID, COL_SESSION, COL_DAY, COL_START, "start_hour",
        COL_WKND, COL_HR, COL_SLEEP, COL_STEPS]
out = df[cols]
out.to_parquet(CLEAN_PARQUET, index=False)
log.info(f"Saved → {CLEAN_PARQUET}  ({len(out):,} rows)")
print(f"\nSaved clean data → {CLEAN_PARQUET}")
print(f"Output shape: {out.shape}")